In [22]:
import json
import shutil
import datetime
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
with open("../BD/JSON/pruebas_de_campeonatos.json", "r", encoding="utf-8") as f:
    pruebas_de_campeonatos = json.load(f)
# with open("../BD/JSON/nuevas_pruebas_de_campeonatos.json", "r", encoding="utf-8") as f:
#     nuevas_pruebas_de_campeonatos = json.load(f)

# pruebas_de_campeonatos = pruebas_de_campeonatos + nuevas_pruebas_de_campeonatos

# with open("../BD/JSON/pruebas_de_campeonatos.json", "w", encoding="utf-8") as f:
#     json.dump(pruebas_de_campeonatos, f, ensure_ascii=False, indent=4)


In [3]:
with open("../BD/JSON/resultados_de_pruebas.json", "r", encoding="utf-8") as f:
    resultados_de_pruebas = json.load(f)
    
# with open("../BD/JSON/nuevos_resultados_de_pruebas.json", "r", encoding="utf-8") as f:
#     nuevos_resultados_de_pruebas = json.load(f)

# resultados_de_pruebas = resultados_de_pruebas + nuevos_resultados_de_pruebas

# with open("../BD/JSON/resultados_de_pruebas.json", "w", encoding="utf-8") as f:
#     json.dump(resultados_de_pruebas, f, ensure_ascii=False, indent=4)

In [4]:
pruebas_from_resultados = pd.DataFrame()
pruebas_from_resultados["id_prueba"] = [prueba["id"] for prueba in resultados_de_pruebas]
pruebas_from_resultados["nombre"] = [prueba["nombre_prueba"] for prueba in resultados_de_pruebas]
pruebas_from_resultados["fecha"] = [prueba["fecha"] for prueba in resultados_de_pruebas]
pruebas_from_resultados["hora"] = [prueba["hora"] for prueba in resultados_de_pruebas]
pruebas_from_resultados["genero"] = [prueba["genero"] for prueba in resultados_de_pruebas]
pruebas_from_resultados["categorias"] = [",".join(prueba["categorias"]) for prueba in resultados_de_pruebas]
pruebas_from_resultados["prueba_padre"] = [None for prueba in resultados_de_pruebas]
pruebas_from_resultados["timestamp"] = [prueba["timestamp"] for prueba in resultados_de_pruebas]

# Convertir fecha de formato string dd/mm/yyyy a formato YYYY-MM-DD y hora de formato string HH:MM a formato HH:MM:SS
pruebas_from_resultados["fecha"] = pd.to_datetime(pruebas_from_resultados["fecha"], errors="coerce", format="%d/%m/%Y").dt.date
pruebas_from_resultados["hora"] = pd.to_datetime(pruebas_from_resultados["hora"], errors="coerce").dt.time

# pruebas_from_resultados.head()
pruebas_from_resultados = pruebas_from_resultados.sort_values("id_prueba").reset_index(drop=True)
pruebas_from_resultados.to_csv("../BD/Tablas/pruebas.csv", index=False, encoding="utf-8")



C:\Users\juanj\AppData\Local\Temp\ipykernel_3692\3413548535.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pruebas_from_resultados["hora"] = pd.to_datetime(pruebas_from_resultados["hora"], errors="coerce").dt.time


In [5]:
pruebas_from_campeonatos = pd.DataFrame()
pruebas_from_campeonatos["id_prueba"] = [prueba["id"] for campeonato in pruebas_de_campeonatos for prueba in campeonato["pruebas"]]
pruebas_from_campeonatos["id_prueba"] = pruebas_from_campeonatos["id_prueba"].astype(int)
pruebas_from_campeonatos["id_campeonato"] = [campeonato["id"] for campeonato in pruebas_de_campeonatos for prueba in campeonato["pruebas"]]
pruebas_from_campeonatos["id_campeonato"] = pruebas_from_campeonatos["id_campeonato"].astype(int)
pruebas_from_campeonatos["nombre_campeonato"] = [campeonato["nombre_campeonato"] for campeonato in pruebas_de_campeonatos for prueba in campeonato["pruebas"]]
pruebas_from_campeonatos["nombre_torneo"] = [campeonato["nombre_torneo"] for campeonato in pruebas_de_campeonatos for prueba in campeonato["pruebas"]]
pruebas_from_campeonatos = pruebas_from_campeonatos.drop_duplicates(subset=["id_prueba"], keep="first").sort_values(by="id_prueba")
pruebas_from_campeonatos


,id_prueba,id_campeonato,nombre_campeonato,nombre_torneo
32625,11,1,Rolf Hoppe 2017,Escolar Atlético Francés
32628,12,1,Rolf Hoppe 2017,Escolar Atlético Francés
32647,13,1,Rolf Hoppe 2017,Escolar Atlético Francés
32691,14,1,Rolf Hoppe 2017,Escolar Atlético Francés
32660,16,1,Rolf Hoppe 2017,Escolar Atlético Francés
...,...,...,...,...
49477,51954,1694,1º Fecha 2026,Fenaude RM
49481,51955,1694,1º Fecha 2026,Fenaude RM
49476,51956,1694,1º Fecha 2026,Fenaude RM
49478,51957,1694,1º Fecha 2026,Fenaude RM


In [6]:
campeonatos_df = pruebas_from_campeonatos.drop_duplicates(subset=["id_campeonato"], keep="first")[["id_campeonato", "nombre_campeonato", "nombre_torneo"]].sort_values(by="id_campeonato")
campeonatos_df.to_csv("../BD/Tablas/campeonatos.csv", index=False, encoding="utf-8")

In [7]:
pruebas_campeonatos = pd.merge(pruebas_from_resultados, pruebas_from_campeonatos, on="id_prueba", how="left")
print(f"Pruebas sin campeonato: {len(pruebas_campeonatos[pd.isna(pruebas_campeonatos['id_campeonato'])])}")
for i in range(len(pruebas_campeonatos) - 1):
    if pd.isna(pruebas_campeonatos.iloc[i]["id_campeonato"]):
        if (pruebas_campeonatos.iloc[i]["nombre"] == pruebas_campeonatos.iloc[i + 1]["nombre"] and
            pruebas_campeonatos.iloc[i]["genero"] == pruebas_campeonatos.iloc[i + 1]["genero"] and
            pruebas_campeonatos.iloc[i]["categorias"] == pruebas_campeonatos.iloc[i + 1]["categorias"]):

            pruebas_campeonatos.at[i, "id_campeonato"] = pruebas_campeonatos.iloc[i + 1]["id_campeonato"]
            pruebas_campeonatos.at[i, "nombre_campeonato"] = pruebas_campeonatos.iloc[i + 1]["nombre_campeonato"]
            pruebas_campeonatos.at[i, "nombre_torneo"] = pruebas_campeonatos.iloc[i + 1]["nombre_torneo"]
            pruebas_campeonatos.at[i, "prueba_padre"] = pruebas_campeonatos.iloc[i + 1]["id_prueba"]

            if pruebas_campeonatos.iloc[i + 1]['id_prueba'] - pruebas_campeonatos.iloc[i]['id_prueba'] > 1:
                print(f"Advertencia: Las pruebas con id {pruebas_campeonatos.iloc[i]['id_prueba']} y {pruebas_campeonatos.iloc[i + 1]['id_prueba']} no son consecutivas, por lo que es posible que no correspondan al mismo campeonato.")
            if pd.isna(pruebas_campeonatos.iloc[i + 1]["id_campeonato"]):
                print("Advertencia: Se asignó la prueba con id {} como padre la prueba con id {}, pero esta tampoco tiene campeonato asignado.".format(pruebas_campeonatos.iloc[i+1]["id_prueba"], pruebas_campeonatos.iloc[i]["id_prueba"]))



asignaciones_realizadas = pruebas_campeonatos[pd.notna(pruebas_campeonatos["prueba_padre"])]
print(f"Se realizaron {len(asignaciones_realizadas)} asignaciones de padre")
pruebas_sin_campeonato_ni_padre = pruebas_campeonatos[pd.isna(pruebas_campeonatos["id_campeonato"]) & pd.isna(pruebas_campeonatos["prueba_padre"])]
print(f"Pruebas sin campeonato ni padre después de asignar campeonatos a pruebas consecutivas: {len(pruebas_sin_campeonato_ni_padre)}")


Pruebas sin campeonato: 5424
Advertencia: Las pruebas con id 5339 y 5342 no son consecutivas, por lo que es posible que no correspondan al mismo campeonato.
Advertencia: Las pruebas con id 6244 y 6247 no son consecutivas, por lo que es posible que no correspondan al mismo campeonato.
Advertencia: Se asignó la prueba con id 28266 como padre la prueba con id 28265, pero esta tampoco tiene campeonato asignado.
Advertencia: Las pruebas con id 32451 y 32454 no son consecutivas, por lo que es posible que no correspondan al mismo campeonato.
Advertencia: Las pruebas con id 32946 y 32949 no son consecutivas, por lo que es posible que no correspondan al mismo campeonato.
Se realizaron 3873 asignaciones de padre
Pruebas sin campeonato ni padre después de asignar campeonatos a pruebas consecutivas: 1551


In [8]:
# Agrupar ids consecutivos
pruebas_sin_campeonato_ni_padre["grupo"] = (pruebas_sin_campeonato_ni_padre["id_prueba"].diff() != 1).cumsum()
grupos = pruebas_sin_campeonato_ni_padre.groupby("grupo").agg({"id_prueba": ["min", "max", "size"]})
# La prueba padre de las pruebas combinadas es el min-1 del grupo 
pruebas_padre_tentativas = grupos.apply(lambda x: x[("id_prueba", "min")] - 1, axis=1)
# Añadir pruebas padre a grupos
grupos["prueba_padre_tentativa"] = pruebas_padre_tentativas
grupos["nombre_prueba_padre_tentativa"] = grupos["prueba_padre_tentativa"].apply(lambda x: pruebas_from_resultados[pruebas_from_resultados["id_prueba"] == x]["nombre"].values[0] if x in pruebas_from_resultados["id_prueba"].values else None)

nombres_pruebas_combinadas = {10: ["Decatlón"], 8: ["Octatlón"], 7: ["Heptatlón"], 6: ["Hexatlón"], 5: ["Pentatlón"], 4: ["Tetratlón", "4 Estaciones"], 3: ["Triatlón", "Tritlón"], 2: ["Duatlón"]}
for row in grupos[grupos[("id_prueba", "size")] > 1].itertuples():
    min = int(row[1])
    max = int(row[2])
    size = int(row[3])
    id_prueba_padre_tentativa = int(row[4])
    nombre_prueba_padre_tentativa = str(row[5])
    if pd.notna(id_prueba_padre_tentativa):
        if any(nombre in nombre_prueba_padre_tentativa for nombre in nombres_pruebas_combinadas.get(size, [])):
            pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"].between(min, max), "prueba_padre"] = id_prueba_padre_tentativa


C:\Users\juanj\AppData\Local\Temp\ipykernel_3692\464500721.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pruebas_sin_campeonato_ni_padre["grupo"] = (pruebas_sin_campeonato_ni_padre["id_prueba"].diff() != 1).cumsum()


In [9]:
asignaciones_realizadas = pruebas_campeonatos[pd.notna(pruebas_campeonatos["prueba_padre"])]
print(f"Se realizaron {len(asignaciones_realizadas)} asignaciones de padres a pruebas combinadas")
pruebas_sin_campeonato_ni_prueba_padre = pruebas_campeonatos[pd.isna(pruebas_campeonatos["id_campeonato"]) & pd.isna(pruebas_campeonatos["prueba_padre"])]
print(f"Pruebas sin campeonato ni padre después de asignar padres a pruebas combinadas: {len(pruebas_sin_campeonato_ni_prueba_padre)}")

Se realizaron 5163 asignaciones de padres a pruebas combinadas
Pruebas sin campeonato ni padre después de asignar padres a pruebas combinadas: 261


In [10]:
# Asignar campeonatos de padres a pruebas hijas
for i in range(len(pruebas_campeonatos)):
    if pd.isna(pruebas_campeonatos.iloc[i]["id_campeonato"]) and pd.notna(pruebas_campeonatos.iloc[i]["prueba_padre"]):
        prueba_padre_id = pruebas_campeonatos.iloc[i]["prueba_padre"]
        prueba_padre = pruebas_campeonatos[pruebas_campeonatos["id_prueba"] == prueba_padre_id]
        if not prueba_padre.empty and pd.notna(prueba_padre.iloc[0]["id_campeonato"]):
            pruebas_campeonatos.at[i, "id_campeonato"] = prueba_padre.iloc[0]["id_campeonato"]
            pruebas_campeonatos.at[i, "nombre_campeonato"] = prueba_padre.iloc[0]["nombre_campeonato"]
            pruebas_campeonatos.at[i, "nombre_torneo"] = prueba_padre.iloc[0]["nombre_torneo"]

asignaciones_realizadas = pruebas_campeonatos[pd.notna(pruebas_campeonatos["prueba_padre"])]
print(f"Se realizaron {len(asignaciones_realizadas)} asignaciones de campeonato después de asignar campeonatos de padres a pruebas hijas")
pruebas_sin_campeonato = pruebas_campeonatos[pd.isna(pruebas_campeonatos["id_campeonato"])]
print(f"Pruebas sin campeonato después de asignar campeonatos de padres a pruebas hijas: {len(pruebas_sin_campeonato)}")


Se realizaron 5163 asignaciones de campeonato después de asignar campeonatos de padres a pruebas hijas
Pruebas sin campeonato después de asignar campeonatos de padres a pruebas hijas: 262


In [11]:
forward_filled = pruebas_campeonatos[["id_campeonato", "nombre_campeonato", "nombre_torneo"]].ffill()
backward_filled = pruebas_campeonatos[["id_campeonato", "nombre_campeonato", "nombre_torneo"]].bfill()
mask = (forward_filled["id_campeonato"] == backward_filled["id_campeonato"])
pruebas_campeonatos.loc[mask, ["id_campeonato", "nombre_campeonato", "nombre_torneo"]] = forward_filled.loc[mask, ["id_campeonato", "nombre_campeonato", "nombre_torneo"]]
pruebas_sin_campeonato = pruebas_campeonatos[pd.isna(pruebas_campeonatos["id_campeonato"])]
print(f"Pruebas sin campeonato después de forward fill y backward fill: {len(pruebas_sin_campeonato)}")

Pruebas sin campeonato después de forward fill y backward fill: 56


In [12]:
caso = 28266
margen = 3
pruebas_campeonatos[pruebas_campeonatos["id_prueba"].between(caso - margen, caso + margen)].sort_values("id_prueba")

,id_prueba,nombre,fecha,hora,genero,categorias,prueba_padre,timestamp,id_campeonato,nombre_campeonato,nombre_torneo
24545,28263,200 Metros Planos,2024-03-03,13:40:00,Hombres,Adulto,None,2026-06-05 00:00:00,930.0,Marzo 2024,Meeting UC
24546,28264,Salto Alto,2024-03-03,13:20:00,Hombres,Adulto,None,2026-06-05 00:00:00,930.0,Marzo 2024,Meeting UC
24547,28265,60 Metros Vallas,2024-02-24,09:00:00,Hombres,Infantil,28266,2026-06-15 22:00:56,NaN,NaN,NaN
24548,28266,60 Metros Vallas,2024-02-24,12:00:00,Hombres,Infantil,None,2026-06-15 22:00:57,NaN,NaN,NaN
24549,28267,5.000 Metros Planos,2024-03-02,11:00:00,Mujeres,"TodoCompetidor,U16,U18,U20",None,2026-06-05 00:00:00,991.0,Torneo Atlético Raúl Guarda 2024,Torneo Atlético Raúl Guarda
24550,28268,100 Metros Planos,2024-03-02,11:25:00,Mujeres,"TodoCompetidor,U16,U18,U20",28269,2026-06-05 00:00:00,991.0,Torneo Atlético Raúl Guarda 2024,Torneo Atlético Raúl Guarda
24551,28269,100 Metros Planos,2024-03-02,12:40:00,Mujeres,"TodoCompetidor,U16,U18,U20",None,2026-06-05 00:00:00,991.0,Torneo Atlético Raúl Guarda 2024,Torneo Atlético Raúl Guarda


In [13]:
# Asignacion manual de campeonatos
casos_puntuales = [(5792, 167), (10907, 414), (20539, 745), (20706, 745), (20780, 750),
                   (30781, 954), (30785, 954), (30787, 954), (30789, 954), (30791, 954)]
for caso, campeonato_asignado in casos_puntuales:
    pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == caso, "id_campeonato"] = campeonatos_df[campeonatos_df["id_campeonato"] == campeonato_asignado]["id_campeonato"].values[0]
    pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == caso, "nombre_campeonato"] = campeonatos_df[campeonatos_df["id_campeonato"] == campeonato_asignado]["nombre_campeonato"].values[0]
    pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == caso, "nombre_torneo"] = campeonatos_df[campeonatos_df["id_campeonato"] == campeonato_asignado]["nombre_torneo"].values[0]


In [14]:
# Asignacion manual de padres

pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 999, "prueba_padre"] = 1033
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 3046, "prueba_padre"] = 3768

for i in range(5982, 5988):
    pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == i, "prueba_padre"] = 5981
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 5988, "prueba_padre"] = 6086
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 5992, "prueba_padre"] = 6039
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 5994, "prueba_padre"] = 6041
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 5996, "prueba_padre"] = 6043
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 5998, "prueba_padre"] = 6045
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 6000, "prueba_padre"] = 6047
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 6002, "prueba_padre"] = 6049
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 9494, "prueba_padre"] = 10141
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 10906, "prueba_padre"] = 10905
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 16467, "prueba_padre"] = 16468
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 18457, "prueba_padre"] = 18608
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 18458, "prueba_padre"] = 18610
for i in range(20817, 20820):
    pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == i, "prueba_padre"] = 20809
for i in range(26750, 26754):
    pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == i, "prueba_padre"] = i + 8
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 33334, "prueba_padre"] = 33335
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34351, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34352, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34353, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34355, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34356, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34357, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 34362, "prueba_padre"] = 34350
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 37619, "prueba_padre"] = 37775
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 37620, "prueba_padre"] = 37773
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 37890, "prueba_padre"] = 37891
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 38160, "prueba_padre"] = 38161
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 39397, "prueba_padre"] = 39405
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 39398, "prueba_padre"] = 39406
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 47692, "prueba_padre"] = 47693
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 51458, "prueba_padre"] = 51459
pruebas_campeonatos.loc[pruebas_campeonatos["id_prueba"] == 51489, "prueba_padre"] = 51490

pruebas_campeonatos[pd.isna(pruebas_campeonatos["id_campeonato"]) & pd.isna(pruebas_campeonatos["prueba_padre"])].reset_index(drop=True)

,id_prueba,nombre,fecha,hora,genero,categorias,prueba_padre,timestamp,id_campeonato,nombre_campeonato,nombre_torneo
0,28266,60 Metros Vallas,2024-02-24,12:00:00,Hombres,Infantil,None,2026-06-15 22:00:57,NaN,NaN,NaN


In [15]:
# Asignar campeonatos de padres a pruebas hijas después de asignar padres manualmente
for i in range(len(pruebas_campeonatos)):
    if pd.isna(pruebas_campeonatos.iloc[i]["id_campeonato"]) and pd.notna(pruebas_campeonatos.iloc[i]["prueba_padre"]):
        prueba_padre_id = pruebas_campeonatos.iloc[i]["prueba_padre"]
        prueba_padre = pruebas_campeonatos[pruebas_campeonatos["id_prueba"] == prueba_padre_id]
        if not prueba_padre.empty and pd.notna(prueba_padre.iloc[0]["id_campeonato"]):
            pruebas_campeonatos.at[i, "id_campeonato"] = prueba_padre.iloc[0]["id_campeonato"]
            pruebas_campeonatos.at[i, "nombre_campeonato"] = prueba_padre.iloc[0]["nombre_campeonato"]
            pruebas_campeonatos.at[i, "nombre_torneo"] = prueba_padre.iloc[0]["nombre_torneo"]

pruebas_sin_campeonato = pruebas_campeonatos[pd.isna(pruebas_campeonatos["id_campeonato"])]
print(f"Pruebas sin campeonato después de asignar campeonatos de padres a pruebas hijas: {len(pruebas_sin_campeonato)}")
pruebas_sin_campeonato

Pruebas sin campeonato después de asignar campeonatos de padres a pruebas hijas: 2


,id_prueba,nombre,fecha,hora,genero,categorias,prueba_padre,timestamp,id_campeonato,nombre_campeonato,nombre_torneo
24547,28265,60 Metros Vallas,2024-02-24,09:00:00,Hombres,Infantil,28266,2026-06-15 22:00:56,NaN,NaN,NaN
24548,28266,60 Metros Vallas,2024-02-24,12:00:00,Hombres,Infantil,None,2026-06-15 22:00:57,NaN,NaN,NaN


In [16]:
pruebas_campeonatos["id_campeonato"] = pruebas_campeonatos["id_campeonato"].astype("Int64")
pruebas_campeonatos["prueba_padre"] = pruebas_campeonatos["prueba_padre"].astype("Int64")
pruebas_campeonatos["id_prueba"] = pruebas_campeonatos["id_prueba"].astype("Int64")
pruebas_campeonatos[["id_prueba", "id_campeonato", "prueba_padre"]].to_csv("../BD/Tablas/pruebas_campeonatos.csv", index=False, encoding="utf-8")

In [17]:
# Elementos de A que no estan en B
A = pruebas_from_campeonatos["id_prueba"]
B = pruebas_from_resultados["id_prueba"]
A_y_B = sorted(set(A) & set(B))
A_o_B = sorted(set(A) | set(B))
A_no_en_B = sorted(set(A) - set(B))
B_no_en_A = sorted(set(B) - set(A))

print(f"Elementos en A: {len(A)}")
print(f"Elementos en B: {len(B)}")
print(f"Intersección A y B: {len(A_y_B)}")
print(f"Unión A y B: {len(A_o_B)}")
print(f"Elementos en A pero no en B: {len(A_no_en_B)}")
print(f"Elementos en B pero no en A: {len(B_no_en_A)}")
# Pruebas desiertas, tablas mal configuradas, pruebas que no se realizaron, etc.
# No capturadas por el scraper 3
print(f"IDs de pruebas con campeonatos pero no en resultados: {A_no_en_B}")
# Series previas a la final
# No capturadas por el scraper 2
print(f"IDs de pruebas con resultados pero no en campeonatos: {B_no_en_A}")

Elementos en A: 42110
Elementos en B: 47534
Intersección A y B: 42110
Unión A y B: 47534
Elementos en A pero no en B: 0
Elementos en B pero no en A: 5424
IDs de pruebas con campeonatos pero no en resultados: []
IDs de pruebas con resultados pero no en campeonatos: [10, 15, 19, 23, 35, 37, 39, 45, 87, 92, 100, 102, 110, 115, 127, 130, 142, 145, 155, 160, 171, 173, 180, 184, 195, 200, 202, 204, 220, 224, 238, 240, 247, 255, 258, 263, 280, 284, 287, 290, 307, 310, 323, 326, 334, 343, 347, 353, 370, 375, 377, 379, 399, 402, 417, 425, 427, 429, 434, 436, 438, 440, 461, 464, 466, 468, 473, 475, 478, 484, 506, 509, 513, 516, 525, 530, 546, 550, 553, 558, 560, 562, 564, 566, 578, 580, 582, 584, 589, 592, 595, 598, 646, 653, 655, 659, 661, 667, 669, 672, 684, 686, 689, 701, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 758, 759, 760, 761, 762, 763, 765, 766, 767, 768, 769, 770, 771, 773, 774, 775, 776, 777, 778, 792, 795, 797, 801, 809, 811, 813, 815, 819, 821, 827, 830, 848, 850, 864, 868,

In [18]:
df_resultados = pd.DataFrame()
df_resultados["id_prueba"] = [prueba["id"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["id_atleta"] = [resultado["id_atleta"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["atleta"] = [resultado["atleta"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["club"] = [resultado["club"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["pista"] = [resultado["pista"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["resultado"] = [resultado["resultado"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["serie"] = [resultado["serie"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]
df_resultados["viento"] = [resultado["viento"] for prueba in resultados_de_pruebas for resultado in prueba["resultados"]]

In [19]:
resultados_de_atletas = df_resultados[df_resultados["id_atleta"].notnull()]
prioridad_resultado = {"DNS": 2, "DNF": 1}
resultados_de_atletas["_prioridad"] = resultados_de_atletas["resultado"].map(
    lambda x: prioridad_resultado.get(x, 0)  # 0 = marca válida (máxima prioridad)
)
# Ordenar por prioridad ascendente y eliminar duplicados
resultados_de_atletas_sin_duplicados = (
    resultados_de_atletas
    .sort_values("_prioridad")
    .drop_duplicates(subset=["id_prueba", "id_atleta"], keep="first")
    .drop(columns="_prioridad")
    .sort_values("id_prueba")
)

resultados_de_atletas_sin_duplicados["id_atleta"] = resultados_de_atletas_sin_duplicados["id_atleta"].astype(int)
resultados_de_atletas_sin_duplicados.to_csv("../BD/Tablas/resultados_atletas.csv", index=False, encoding="utf-8")

C:\Users\juanj\AppData\Local\Temp\ipykernel_3692\37206835.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  resultados_de_atletas["_prioridad"] = resultados_de_atletas["resultado"].map(


In [20]:
resultados_de_relevos = df_resultados[df_resultados["id_atleta"].isnull()]
resultados_de_relevos_sin_duplicados = resultados_de_relevos.drop_duplicates(subset=["id_prueba", "club", "pista", "resultado"], keep="first")

resultados_de_relevos_sin_duplicados.to_csv("../BD/Tablas/resultados_relevos.csv", index=False, encoding="utf-8")

In [23]:
# Crear backup de la bd
origen = Path("../BD")
destino = Path("../BD_Backup")
fecha_hora_actual = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
destino_con_fecha = destino / f"Backup_{fecha_hora_actual}"

shutil.copytree(origen, destino_con_fecha)

WindowsPath('../BD_Backup/Backup_20260617_235733')